
#!/bin/bash

read -rsp "Enter fresh MFA OTP: " OTP
echo

RESPONSE=$(aws sts get-session-token \
    --serial-number arn:aws:iam::724446904294:mfa/POCO_NOVA_Opstree \
    --token-code "$OTP" \
    --duration-seconds 3600 \
    --output json)

unset OTP

if echo "$RESPONSE" | jq -e '.Credentials.AccessKeyId' >/dev/null; then

    aws configure set aws_access_key_id \
        "$(echo "$RESPONSE" | jq -r '.Credentials.AccessKeyId')" \
        --profile mfa-session

    aws configure set aws_secret_access_key \
        "$(echo "$RESPONSE" | jq -r '.Credentials.SecretAccessKey')" \
        --profile mfa-session

    aws configure set aws_session_token \
        "$(echo "$RESPONSE" | jq -r '.Credentials.SessionToken')" \
        --profile mfa-session

    echo "MFA session configured successfully."

    aws sts get-caller-identity --profile mfa-session

else
    echo "MFA authentication failed. Check OTP and try again."
fi

unset RESPONSE


In [1]:
%pip install boto3 openpyxl

  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 1.2 MB/s  0:00:13 eta 0:00:010:00:01m
Using cached jmespath-1.1.0-py3-none-any.whl (20 kB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 4.0.0
    Uninstalling python-dateutil-4.0.0:
      Successfully uninstalled python-dateutil-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [boto3]━━━━━━━━━━━━ 4/7 [botocore]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
mukesh@mukesharma:~$ read -rsp "Enter fresh MFA OTP: " OTP
echo

RESPONSE=$(aws sts get-session-token \
  --serial-number arn:aws:iam::724446904294:mfa/POCO_NOVA_Opstree \
  --token-code "$OTP" \
  --duration-seconds 3600 \
  --output json)

unset OTP

if echo "$RESPONSE" | jq -e '.Credentials.AccessKeyId' >/dev/null; then

    aws configure set aws_access_key_id \
      "$(echo "$RESPONSE" | jq -r '.Credentials.AccessKeyId')" \
      --profile mfa-session

    aws configure set aws_secret_access_key \
      "$(echo "$RESPONSE" | jq -r '.Credentials.SecretAccessKey')" \
      --profile mfa-session

    aws configure set aws_session_token \
      "$(echo "$RESPONSE" | jq -r '.Credentials.SessionToken')" \
fi  unset RESPONSEentication failed"cessfully!"e mfa-session
Enter fresh MFA OTP: 
MFA profile configured successfully!
mukesh@mukesharma:~$ aws sts get-caller-identity --profile mfa-session
{
    "UserId": "AIDA2RLDGP7TLWUKUU3TJ",
    "Account": "724446904294",
    "Arn": "arn:aws:iam::724446904294:user/Mukesh"
}
mukesh@mukesharma:~$ aws ce get-cost-and-usage \
  --time-period Start=2026-10-01,End=2026-11-01 \
  --granularity MONTHLY \
  --metrics UnblendedCost \
  --profile mfa-session
{
    "ResultsByTime": [
        {
            "TimePeriod": {
                "Start": "2026-10-01",
                "End": "2026-11-01"
            },
            "Total": {
                "UnblendedCost": {
                    "Amount": "-0.0000150388",
                    "Unit": "USD"
                }
            },
            "Groups": [],
            "Estimated": false
        }
    ],
    "DimensionValueAttributes": []
}


In [1]:
import argparse
import boto3
import pandas as pd
import os
from datetime import datetime, timedelta, timezone
from botocore.exceptions import ClientError, BotoCoreError
import json
import re

# ============================================================
# CONFIGURATION
# ============================================================

DEFAULT_PROFILE = "mfa-session"
DEFAULT_REGION = "ap-south-1"
DEFAULT_DAYS = 90


# ============================================================
# AWS SESSION
# ============================================================

def create_session(profile, region):

    session = boto3.Session(
        profile_name=profile,
        region_name=region
    )

    sts = session.client("sts")

    identity = sts.get_caller_identity()

    print("\nAWS Account:", identity["Account"])
    print("Caller ARN:", identity["Arn"])

    return session, identity


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_cost_explorer(session):

    return session.client(
        "ce",
        region_name="us-east-1"
    )


def get_cost_amount(metrics):

    metric = metrics.get("UnblendedCost", {})

    return {
        "Cost": float(metric.get("Amount", 0)),
        "Currency": metric.get("Unit", "USD")
    }


def get_cost_data(
    ce,
    start_date,
    end_date,
    granularity,
    group_by,
    sheet_name
):

    rows = []

    try:

        params = {

            "TimePeriod": {
                "Start": start_date,
                "End": end_date
            },

            "Granularity": granularity,

            "Metrics": [
                "UnblendedCost"
            ],

            "GroupBy": group_by

        }

        while True:

            response = ce.get_cost_and_usage(**params)

            for period in response.get("ResultsByTime", []):

                period_start = period.get(
                    "TimePeriod", {}
                ).get("Start", "")

                period_end = period.get(
                    "TimePeriod", {}
                ).get("End", "")

                for group in period.get("Groups", []):

                    keys = group.get("Keys", [])

                    cost_info = get_cost_amount(
                        group.get("Metrics", {})
                    )

                    row = {

                        "Period Start": period_start,

                        "Period End": period_end,

                        "Cost": cost_info["Cost"],

                        "Currency": cost_info["Currency"]

                    }

                    for index, dimension in enumerate(group_by):

                        key = dimension["Key"]

                        row[key] = (
                            keys[index]
                            if index < len(keys)
                            else "Unknown"
                        )

                    rows.append(row)

            # Handle API pagination manually

            next_token = response.get("NextPageToken")

            if not next_token:
                break

            params["NextPageToken"] = next_token

    except (ClientError, BotoCoreError) as exc:

        print(f"[ERROR] {sheet_name}: {exc}")

    return rows
# ============================================================
# 1. 90-DAY SERVICE COST
# ============================================================

def get_service_cost(ce, start, end):

    print("\n[1] Fetching 90-day service cost...")

    return get_cost_data(

        ce,
        start,
        end,

        "MONTHLY",

        [
            {
                "Type": "DIMENSION",
                "Key": "SERVICE"
            }
        ],

        "Service Cost"

    )


# ============================================================
# 2. DAILY COST TREND
# ============================================================

def get_daily_cost(ce, start, end):

    print("\n[2] Fetching daily cost trend...")

    return get_cost_data(

        ce,
        start,
        end,

        "DAILY",

        [
            {
                "Type": "DIMENSION",
                "Key": "SERVICE"
            }
        ],

        "Daily Cost"

    )


# ============================================================
# 3. SERVICE + USAGE TYPE COST
# ============================================================

def get_usage_breakdown(ce, start, end):

    print("\n[3] Fetching usage type breakdown...")

    return get_cost_data(

        ce,
        start,
        end,

        "MONTHLY",

        [
            {
                "Type": "DIMENSION",
                "Key": "SERVICE"
            },
            {
                "Type": "DIMENSION",
                "Key": "USAGE_TYPE"
            }
        ],

        "Usage Breakdown"

    )


# ============================================================
# 4. SERVICE + OPERATION COST
# ============================================================

def get_operation_breakdown(ce, start, end):

    print("\n[4] Fetching operation breakdown...")

    return get_cost_data(

        ce,
        start,
        end,

        "MONTHLY",

        [
            {
                "Type": "DIMENSION",
                "Key": "SERVICE"
            },
            {
                "Type": "DIMENSION",
                "Key": "OPERATION"
            }
        ],

        "Operation Breakdown"

    )


# ============================================================
# 5. SERVICE + REGION COST
# ============================================================

def get_region_cost(ce, start, end):

    print("\n[5] Fetching regional cost...")

    return get_cost_data(

        ce,
        start,
        end,

        "MONTHLY",

        [
            {
                "Type": "DIMENSION",
                "Key": "SERVICE"
            },
            {
                "Type": "DIMENSION",
                "Key": "REGION"
            }
        ],

        "Region Breakdown"

    )


# ============================================================
# 6. COST FORECAST
# ============================================================

def get_cost_forecast(ce):

    print("\n[6] Fetching cost forecast...")

    rows = []

    today = datetime.now(timezone.utc).date()

    first_day_next_month = (
        today.replace(day=1) + timedelta(days=32)
    ).replace(day=1)

    first_day_following_month = (
        first_day_next_month + timedelta(days=32)
    ).replace(day=1)

    try:

        response = ce.get_cost_forecast(

            TimePeriod={

                "Start": first_day_next_month.isoformat(),

                "End": first_day_following_month.isoformat()

            },

            Metric="UNBLENDED_COST",

            Granularity="MONTHLY"

        )

        for item in response.get("ForecastResultsByTime", []):

            mean_value = item.get(
                "MeanValue", 0
            )

            rows.append({

                "Forecast Start": item.get(
                    "TimePeriod", {}
                ).get("Start", ""),

                "Forecast End": item.get(
                    "TimePeriod", {}
                ).get("End", ""),

                "Estimated Cost": float(mean_value),

                "Currency": "USD",

                "Explanation": (
                    "AWS Cost Explorer forecast based on "
                    "historical spending patterns."
                )

            })

    except (ClientError, BotoCoreError) as exc:

        print("[WARNING] Forecast unavailable:", exc)

        rows.append({

            "Forecast Start": "",
            "Forecast End": "",
            "Estimated Cost": "",
            "Currency": "",
            "Explanation": str(exc)

        })

    return rows


# ============================================================
# 7. COST EXPLANATION
# ============================================================

def explain_usage(usage_rows):

    print("\n[7] Preparing cost explanations...")

    explanations = {

        "BoxUsage": "EC2 instance running hours.",

        "InstanceUsage": "Compute instance usage.",

        "EBS:VolumeUsage": "EBS storage consumption.",

        "EBS:SnapshotUsage": "EBS snapshot storage.",

        "NatGateway-Hours": "NAT Gateway running hours.",

        "NatGateway-Bytes": "NAT Gateway data processing.",

        "DataTransfer": "Data transfer charges.",

        "TimedStorage": "S3 storage consumption.",

        "Requests": "API or storage request charges.",

        "RDS:InstanceUsage": "RDS database running hours.",

        "RDS:StorageUsage": "RDS database storage.",

        "Lambda-GB-Second": "Lambda memory and execution duration.",

        "Lambda-Requests": "Lambda invocation requests.",

        "Fargate-vCPU-Hours": "Fargate vCPU consumption.",

        "Fargate-GB-Hours": "Fargate memory consumption."

    }

    result = []

    for row in usage_rows:

        usage_type = row.get(
            "USAGE_TYPE", ""
        )

        explanation = "Check the AWS usage type and pricing dimension."

        for key, value in explanations.items():

            if key.lower() in usage_type.lower():

                explanation = value

                break

        result.append({

            "Service": row.get("SERVICE", ""),

            "Usage Type": usage_type,

            "Month": row.get("Period Start", ""),

            "Cost": row.get("Cost", 0),

            "Currency": row.get("Currency", ""),

            "Possible Cost Reason": explanation

        })

    return result


# ============================================================
# 8. CURRENT RESOURCE INVENTORY
# ============================================================

def get_current_resources(session, region):

    print("\n[8] Scanning current AWS resources...")

    resources = []

    def add_resource(
        service,
        resource_type,
        resource_id,
        name,
        state,
        created,
        tags=None
    ):

        tags = tags or {}

        resources.append({

            "Service": service,

            "Resource Type": resource_type,

            "Resource ID": resource_id,

            "Name": name,

            "State": state,

            "Created At": created,

            "Owner Tag": (
                tags.get("Owner")
                or tags.get("owner")
                or tags.get("CreatedBy")
                or ""
            ),

            "Environment Tag": (
                tags.get("Environment")
                or tags.get("environment")
                or ""
            ),

            "Region": region

        })

    # --------------------------------------------------------
    # EC2 INSTANCES
    # --------------------------------------------------------

    try:

        ec2 = session.client(
            "ec2",
            region_name=region
        )

        paginator = ec2.get_paginator(
            "describe_instances"
        )

        for page in paginator.paginate():

            for reservation in page.get("Reservations", []):

                for instance in reservation.get("Instances", []):

                    tags = {
                        item["Key"]: item["Value"]
                        for item in instance.get("Tags", [])
                    }

                    add_resource(

                        "EC2",

                        "Instance",

                        instance.get("InstanceId"),

                        tags.get("Name", ""),

                        instance.get("State", {}).get("Name", ""),

                        instance.get("LaunchTime"),

                        tags

                    )

        # EBS VOLUMES

        for page in ec2.get_paginator(
            "describe_volumes"
        ).paginate():

            for volume in page.get("Volumes", []):

                tags = {
                    item["Key"]: item["Value"]
                    for item in volume.get("Tags", [])
                }

                add_resource(

                    "EBS",

                    "Volume",

                    volume.get("VolumeId"),

                    tags.get("Name", ""),

                    volume.get("State", ""),

                    volume.get("CreateTime"),

                    tags

                )

    except (ClientError, BotoCoreError) as exc:

        print("[WARNING] EC2/EBS scan:", exc)

    # --------------------------------------------------------
    # RDS
    # --------------------------------------------------------

    try:

        rds = session.client(
            "rds",
            region_name=region
        )

        for page in rds.get_paginator(
            "describe_db_instances"
        ).paginate():

            for db in page.get("DBInstances", []):

                arn = db.get("DBInstanceArn")

                tags_response = rds.list_tags_for_resource(
                    ResourceName=arn
                )

                tags = {
                    item["Key"]: item["Value"]
                    for item in tags_response.get("TagList", [])
                }

                add_resource(

                    "RDS",

                    "DB Instance",

                    db.get("DBInstanceIdentifier"),

                    db.get("DBName", ""),

                    db.get("DBInstanceStatus", ""),

                    db.get("InstanceCreateTime"),

                    tags

                )

    except (ClientError, BotoCoreError) as exc:

        print("[WARNING] RDS scan:", exc)

    # --------------------------------------------------------
    # S3
    # --------------------------------------------------------

    try:

        s3 = session.client("s3")

        for bucket in s3.list_buckets().get("Buckets", []):

            bucket_name = bucket.get("Name")

            try:

                tag_response = s3.get_bucket_tagging(
                    Bucket=bucket_name
                )

                tags = {
                    item["Key"]: item["Value"]
                    for item in tag_response.get("TagSet", [])
                }

            except ClientError:

                tags = {}

            add_resource(

                "S3",

                "Bucket",

                bucket_name,

                bucket_name,

                "Active",

                bucket.get("CreationDate"),

                tags

            )

    except (ClientError, BotoCoreError) as exc:

        print("[WARNING] S3 scan:", exc)

    # --------------------------------------------------------
    # LAMBDA
    # --------------------------------------------------------

    try:

        lam = session.client(
            "lambda",
            region_name=region
        )

        for page in lam.get_paginator(
            "list_functions"
        ).paginate():

            for function in page.get("Functions", []):

                arn = function.get("FunctionArn")

                tags = lam.list_tags(
                    Resource=arn
                ).get("Tags", {})

                add_resource(

                    "Lambda",

                    "Function",

                    function.get("FunctionArn"),

                    function.get("FunctionName"),

                    function.get("State", "Active"),

                    function.get("LastModified"),

                    tags

                )

    except (ClientError, BotoCoreError) as exc:

        print("[WARNING] Lambda scan:", exc)

    # --------------------------------------------------------
    # EKS
    # --------------------------------------------------------

    try:

        eks = session.client(
            "eks",
            region_name=region
        )

        for page in eks.get_paginator(
            "list_clusters"
        ).paginate():

            for cluster_name in page.get("clusters", []):

                cluster = eks.describe_cluster(
                    name=cluster_name
                )["cluster"]

                add_resource(

                    "EKS",

                    "Cluster",

                    cluster.get("arn"),

                    cluster_name,

                    cluster.get("status", ""),

                    cluster.get("createdAt"),

                    cluster.get("tags", {})

                )

    except (ClientError, BotoCoreError) as exc:

        print("[WARNING] EKS scan:", exc)

    # --------------------------------------------------------
    # ELB
    # --------------------------------------------------------

    try:

        elb = session.client(
            "elbv2",
            region_name=region
        )

        for page in elb.get_paginator(
            "describe_load_balancers"
        ).paginate():

            for lb in page.get("LoadBalancers", []):

                arn = lb.get("LoadBalancerArn")

                tag_response = elb.describe_tags(
                    ResourceArns=[arn]
                )

                tag_list = tag_response.get(
                    "TagDescriptions", []
                )

                tags = {}

                if tag_list:

                    tags = {
                        item["Key"]: item["Value"]
                        for item in tag_list[0].get("Tags", [])
                    }

                add_resource(

                    "ELB",

                    "Load Balancer",

                    arn,

                    lb.get("LoadBalancerName"),

                    lb.get("State", {}).get("Code", ""),

                    "",

                    tags

                )

    except (ClientError, BotoCoreError) as exc:

        print("[WARNING] ELB scan:", exc)

    return resources

    # ============================================================
# 11. CLOUDTRAIL IAM ATTRIBUTION
# ============================================================

# Targeted management events only.
# CloudTrail Event History supports lookup for the last 90 days.

TARGET_EVENTS = [
    "RunInstances",
    "StartInstances",
    "StopInstances",
    "TerminateInstances",
    "CreateVolume",
    "CreateDBInstance",
    "StartDBInstance",
    "StopDBInstance",
    "DeleteDBInstance",
    "CreateBucket",
    "CreateFunction",
    "UpdateFunctionConfiguration",
    "CreateCluster",
    "DeleteCluster",
    "CreateLoadBalancer",
    "CreateTable",
    "CreateQueue",
    "CreateRepository",
    "CreateSnapshot",
]

def get_enabled_regions(session):

    regions = []

    try:

        ec2 = session.client("ec2")

        response = ec2.describe_regions(
            AllRegions=False
        )

        regions = [
            item["RegionName"]
            for item in response.get("Regions", [])
        ]

    except (ClientError, BotoCoreError) as exc:

        print("[WARNING] Could not list AWS regions:", exc)

    if not regions:
        regions = [session.region_name or DEFAULT_REGION]

    print(f"\nEnabled AWS regions found: {len(regions)}")

    return sorted(set(regions))


def extract_principal(event):

    identity = event.get("userIdentity", {})

    identity_type = identity.get("type", "Unknown")

    principal_arn = identity.get("arn", "")

    principal_id = identity.get("principalId", "")

    user_name = identity.get("userName", "")

    issuer = (
        identity.get("sessionContext", {})
        .get("sessionIssuer", {})
    )

    issuer_arn = issuer.get("arn", "")

    issuer_name = issuer.get("userName", "")

    if identity_type == "IAMUser":

        principal_name = user_name or principal_arn.split("/")[-1]

        principal_type = "IAM User"

    elif identity_type == "AssumedRole":

        principal_name = issuer_name or (
            issuer_arn.split("/")[-1]
            if issuer_arn else principal_arn.split("/")[-1]
        )

        principal_type = "Assumed IAM Role"

    elif identity_type == "Root":

        principal_name = "Root"

        principal_type = "Root Account"

    elif identity_type == "AWSService":

        principal_name = identity.get(
            "invokedBy",
            principal_arn or "AWS Service"
        )

        principal_type = "AWS Service"

    else:

        principal_name = (
            user_name
            or principal_arn
            or principal_id
            or "Unknown"
        )

        principal_type = identity_type

    return {
        "Principal Name": principal_name,
        "Principal Type": principal_type,
        "Principal ARN": principal_arn,
        "Principal ID": principal_id,
        "Session Issuer ARN": issuer_arn,
        "Session Issuer Name": issuer_name,
    }


def collect_event_values(data, values=None):

    if values is None:
        values = set()

    if isinstance(data, dict):

        for key, value in data.items():

            if isinstance(value, (dict, list)):
                collect_event_values(value, values)

            elif isinstance(value, str):

                # Collect IDs, ARNs, resource names and bucket names.
                if (
                    key.lower().endswith("id")
                    or key.lower().endswith("arn")
                    or key.lower() in [
                        "name",
                        "bucket",
                        "bucketname",
                        "resourcename",
                        "functionname",
                        "clustername",
                        "dbinstanceidentifier",
                        "instancename",
                    ]
                ):

                    values.add(value)

    elif isinstance(data, list):

        for item in data:
            collect_event_values(item, values)

    return values


def get_resource_match_keys(resource):

    keys = set()

    for field in [
        "Resource ID",
        "Name",
    ]:

        value = resource.get(field)

        if value:
            keys.add(str(value))

    return keys


def scan_cloudtrail_region(
    session,
    region,
    start_time,
    end_time
):

    print(f"[CloudTrail] Scanning region: {region}")

    rows = []

    try:

        cloudtrail = session.client(
            "cloudtrail",
            region_name=region
        )

    except (ClientError, BotoCoreError) as exc:

        print(f"[WARNING] CloudTrail client {region}: {exc}")

        return rows

    for event_name in TARGET_EVENTS:

        try:

            paginator = cloudtrail.get_paginator(
                "lookup_events"
            )

            pages = paginator.paginate(

                LookupAttributes=[
                    {
                        "AttributeKey": "EventName",
                        "AttributeValue": event_name
                    }
                ],

                StartTime=start_time,

                EndTime=end_time,

                PaginationConfig={
                    "PageSize": 50
                }

            )

            for page in pages:

                for item in page.get("Events", []):

                    raw_event = item.get(
                        "CloudTrailEvent",
                        "{}"
                    )

                    try:

                        event = json.loads(raw_event)

                    except (json.JSONDecodeError, TypeError):

                        continue

                    principal = extract_principal(event)

                    event_values = collect_event_values(event)

                    event_resources = event.get(
                        "resources",
                        []
                    )

                    resource_ids = []

                    for resource_item in event_resources:

                        resource_id = resource_item.get(
                            "resourceName"
                        )

                        if resource_id:
                            resource_ids.append(resource_id)

                    resource_ids.extend(
                        list(event_values)
                    )

                    rows.append({

                        "Event Time": event.get(
                            "eventTime",
                            item.get("EventTime")
                        ),

                        "Event Name": event.get(
                            "eventName",
                            event_name
                        ),

                        "AWS Service": event.get(
                            "eventSource",
                            ""
                        ),

                        "AWS Region": event.get(
                            "awsRegion",
                            region
                        ),

                        "Principal Name": principal["Principal Name"],

                        "Principal Type": principal["Principal Type"],

                        "Principal ARN": principal["Principal ARN"],

                        "Principal ID": principal["Principal ID"],

                        "Session Issuer ARN": principal["Session Issuer ARN"],

                        "Session Issuer Name": principal["Session Issuer Name"],

                        "Source IP": event.get(
                            "sourceIPAddress",
                            ""
                        ),

                        "User Agent": event.get(
                            "userAgent",
                            ""
                        ),

                        "Resource IDs": ", ".join(
                            sorted(set(resource_ids))
                        ),

                        "CloudTrail Event ID": event.get(
                            "eventID",
                            item.get("EventId", "")
                        ),

                        "Read Only": event.get(
                            "readOnly",
                            ""
                        ),

                        "_match_values": event_values,

                    })

        except ClientError as exc:

            error_code = exc.response.get(
                "Error", {}
            ).get("Code", "")

            print(
                f"[WARNING] CloudTrail {region} "
                f"{event_name}: {error_code} - {exc}"
            )

            continue

        except BotoCoreError as exc:

            print(
                f"[WARNING] CloudTrail {region} "
                f"{event_name}: {exc}"
            )

            continue

    return rows


def get_cloudtrail_activity(
    session,
    days
):

    print("\n[11] Scanning targeted CloudTrail IAM activity...")

    end_time = datetime.now(timezone.utc)

    start_time = end_time - timedelta(days=days)

    regions = get_enabled_regions(session)

    all_events = []

    for region in regions:

        region_events = scan_cloudtrail_region(
            session,
            region,
            start_time,
            end_time
        )

        all_events.extend(region_events)

        print(
            f"[CloudTrail] {region}: "
            f"{len(region_events)} targeted events"
        )

    print(
        "\nTotal CloudTrail events collected:",
        len(all_events)
    )

    return all_events


def attribute_resources(resources, cloudtrail_events):

    print("\n[12] Mapping IAM activity to current resources...")

    attribution_rows = []

    timeline_rows = []

    unidentified_rows = []

    events_by_resource = {}

    # Match CloudTrail event values with inventory IDs/names.
    for resource in resources:

        resource_keys = get_resource_match_keys(resource)

        matched_events = []

        for event in cloudtrail_events:

            event_values = event.get(
                "_match_values",
                set()
            )

            event_resource_ids = {
                value.strip()
                for value in event.get(
                    "Resource IDs",
                    ""
                ).split(",")
                if value.strip()
            }

            candidates = event_values.union(
                event_resource_ids
            )

            # Exact match only; avoid guessing based on partial IDs.
            if resource_keys.intersection(candidates):

                matched_events.append(event)

        # Sort oldest to newest.
        matched_events.sort(
            key=lambda item: str(item.get("Event Time", ""))
        )

        resource_id = resource.get("Resource ID", "")

        if matched_events:

            creation_events = [
                event for event in matched_events
                if event.get("Event Name", "").startswith(
                    ("Create", "RunInstances")
                )
            ]

            start_events = [
                event for event in matched_events
                if event.get("Event Name", "").startswith(
                    ("Start",)
                )
            ]

            latest_event = matched_events[-1]

            creator_event = (
                creation_events[0]
                if creation_events
                else None
            )

            starter_event = (
                start_events[-1]
                if start_events
                else None
            )

            creator = (
                extract_principal_from_row(creator_event)
                if creator_event
                else {}
            )

            starter = (
                extract_principal_from_row(starter_event)
                if starter_event
                else {}
            )

            attribution_rows.append({

                "Service": resource.get("Service"),

                "Resource Type": resource.get("Resource Type"),

                "Resource ID": resource_id,

                "Resource Name": resource.get("Name"),

                "Region": resource.get("Region"),

                "Current State": resource.get("State"),

                "Created At (Inventory)": resource.get("Created At"),

                "Creator IAM Principal": (
                    creator.get("Principal Name", "Unknown")
                ),

                "Creator Principal Type": (
                    creator.get("Principal Type", "Unknown")
                ),

                "Creator ARN": (
                    creator.get("Principal ARN", "")
                ),

                "Creator Event": (
                    creator_event.get("Event Name", "")
                    if creator_event else ""
                ),

                "Creator Event Time": (
                    creator_event.get("Event Time", "")
                    if creator_event else ""
                ),

                "Last Starter Principal": (
                    starter.get("Principal Name", "Unknown")
                ),

                "Last Starter ARN": (
                    starter.get("Principal ARN", "")
                ),

                "Last Start Event Time": (
                    starter_event.get("Event Time", "")
                    if starter_event else ""
                ),

                "Latest Activity Principal": (
                    latest_event.get("Principal Name", "Unknown")
                ),

                "Latest Activity": latest_event.get("Event Name"),

                "Latest Activity Time": latest_event.get("Event Time"),

                "Attribution Status": (
                    "Creator Identified"
                    if creator_event
                    else "Activity Found - Creator Not Confirmed"
                ),

                "Cost Attribution": (
                    "Not directly available from Cost Explorer"
                )

            })

        else:

            attribution_rows.append({

                "Service": resource.get("Service"),

                "Resource Type": resource.get("Resource Type"),

                "Resource ID": resource_id,

                "Resource Name": resource.get("Name"),

                "Region": resource.get("Region"),

                "Current State": resource.get("State"),

                "Created At (Inventory)": resource.get("Created At"),

                "Creator IAM Principal": "Unknown",

                "Creator Principal Type": "Unknown",

                "Creator ARN": "",

                "Creator Event": "",

                "Creator Event Time": "",

                "Last Starter Principal": "Unknown",

                "Last Starter ARN": "",

                "Last Start Event Time": "",

                "Latest Activity Principal": "Unknown",

                "Latest Activity": "",

                "Latest Activity Time": "",

                "Attribution Status": "No Matching CloudTrail Event",

                "Cost Attribution": (
                    "Not directly available from Cost Explorer"
                )

            })

            unidentified_rows.append({

                "Service": resource.get("Service"),

                "Resource Type": resource.get("Resource Type"),

                "Resource ID": resource_id,

                "Resource Name": resource.get("Name"),

                "Region": resource.get("Region"),

                "Created At": resource.get("Created At"),

                "Reason": (
                    "No matching targeted CloudTrail event found. "
                    "The event may be older than 90 days, recorded "
                    "in another account, or not covered by the selected "
                    "event list."
                )

            })

    # Create a separate event timeline for all collected events.
    for event in cloudtrail_events:

        timeline_rows.append({

            key: value

            for key, value in event.items()

            if key != "_match_values"

        })

    return (
        attribution_rows,
        timeline_rows,
        unidentified_rows
    )


def extract_principal_from_row(event):

    if not event:
        return {}

    return {

        "Principal Name": event.get("Principal Name", "Unknown"),

        "Principal Type": event.get("Principal Type", "Unknown"),

        "Principal ARN": event.get("Principal ARN", ""),

    }
# ============================================================
# 9. KIRO DIAGNOSTIC
# ============================================================

def kiro_diagnostic(session, identity, region):

    print("\n[9] Checking Kiro-related identity information...")

    rows = []

    arn = identity.get("Arn", "")

    rows.append({

        "Check": "Current AWS Identity",

        "Result": arn,

        "Status": "Retrieved",

        "Explanation": (
            "This identifies the AWS credentials currently used "
            "by the script. It does not prove Kiro subscription assignment."
        )

    })

    rows.append({

        "Check": "AWS Account",

        "Result": identity.get("Account", ""),

        "Status": "Retrieved",

        "Explanation": "AWS account associated with current credentials."

    })

    try:

        sso = session.client(
            "sso-admin",
            region_name=region
        )

        response = sso.list_instances()

        instances = response.get("Instances", [])

        if instances:

            for instance in instances:

                rows.append({

                    "Check": "Identity Center Instance",

                    "Result": instance.get("InstanceArn", ""),

                    "Status": "Visible",

                    "Explanation": (
                        "Identity Center instance is visible. "
                        "This does not confirm Kiro seat assignment."
                    )

                })

        else:

            rows.append({

                "Check": "Identity Center Instance",

                "Result": "No instance returned",

                "Status": "Not Found",

                "Explanation": (
                    "No Identity Center instance was returned "
                    "in this account and region context."
                )

            })

    except (ClientError, BotoCoreError) as exc:

        rows.append({

            "Check": "Identity Center Visibility",

            "Result": str(exc),

            "Status": "Unavailable",

            "Explanation": (
                "The script could not verify Identity Center visibility."
            )

        })

    rows.append({

        "Check": "Kiro Subscription Assignment",

        "Result": "Manual admin-console verification required",

        "Status": "Not Verified",

        "Explanation": (
            "This script cannot conclusively verify Kiro seat assignment. "
            "Check Kiro Users & Groups > Subscriptions, user identity, "
            "subscription status and organization settings."
        )

    })

    return rows


# ============================================================
# 10. MISSING OWNER TAGS
# ============================================================

def get_missing_owner_tags(resources):

    rows = []

    for resource in resources:

        if not resource.get("Owner Tag"):

            rows.append({

                "Service": resource.get("Service"),

                "Resource Type": resource.get("Resource Type"),

                "Resource ID": resource.get("Resource ID"),

                "Name": resource.get("Name"),

                "Region": resource.get("Region"),

                "Issue": "Owner tag missing",

                "Explanation": (
                    "Resource creator cannot be reliably identified "
                    "from current resource tags."
                )

            })

    return rows


# ============================================================
# 11. EXCEL REPORT
# ============================================================
# ============================================================
# 13. EXCEL REPORT
# ============================================================

def generate_report(
    output,
    service_cost,
    daily_cost,
    usage_cost,
    operation_cost,
    region_cost,
    forecast,
    explanations,
    resources,
    missing_tags,
    kiro,
    iam_attribution,
    iam_timeline,
    unidentified_resources
):

    print("\n[13] Generating Excel report...")

    with pd.ExcelWriter(
        output,
        engine="openpyxl"
    ) as writer:

        def save_sheet(name, data):

            df = pd.DataFrame(data)

            if df.empty:

                df = pd.DataFrame({
                    "Information": [
                        "No data returned or access unavailable."
                    ]
                })

            def clean_datetime(value):

                if isinstance(value, pd.Timestamp):

                    if value.tzinfo is not None:
                        return value.tz_localize(None)

                elif isinstance(value, datetime):

                    if value.tzinfo is not None:
                        return value.replace(tzinfo=None)

                return value

            df = df.apply(
                lambda column: column.map(clean_datetime)
            )

            df.to_excel(
                writer,
                sheet_name=name[:31],
                index=False
            )

            print(f"[SUCCESS] Sheet created: {name}")

        sheets = [

            ("90-Day Service Cost", service_cost),

            ("Daily Cost Trend", daily_cost),

            ("Usage Breakdown", usage_cost),

            ("Operation Breakdown", operation_cost),

            ("Region Breakdown", region_cost),

            ("Cost Forecast", forecast),

            ("Cost Explanation", explanations),

            ("Current Resources", resources),

            ("Missing Owner Tags", missing_tags),

            ("Kiro Diagnostic", kiro),

            ("IAM Resource Attribution", iam_attribution),

            ("IAM Activity Timeline", iam_timeline),

            ("Unidentified Resources", unidentified_resources),

        ]

        for sheet_name, sheet_data in sheets:

            save_sheet(
                sheet_name,
                sheet_data
            )

        # Excel formatting
        for worksheet in writer.sheets.values():

            worksheet.freeze_panes = "A2"

            worksheet.auto_filter.ref = worksheet.dimensions

            for column in worksheet.iter_cols():

                header = str(
                    column[0].value or ""
                ).strip().lower()

                if "cost" in header or "amount" in header:

                    for cell in column[1:]:

                        if isinstance(cell.value, (int, float)):

                            cell.number_format = (
                                '$#,##0.00000000;'
                                '[Red]-$#,##0.00000000'
                            )

            for column in worksheet.columns:

                max_length = 0

                column_letter = column[0].column_letter

                for cell in column:

                    try:

                        max_length = max(
                            max_length,
                            len(str(cell.value))
                        )

                    except Exception:
                        pass

                worksheet.column_dimensions[
                    column_letter
                ].width = min(max_length + 3, 55)

    print("\n[SUCCESS] Report generated:", output)

# ============================================================
# MAIN
# ============================================================

# ============================================================
# MAIN
# ============================================================

def main():

    parser = argparse.ArgumentParser(
        description="AWS Cost and IAM Resource Audit"
    )

    parser.add_argument(
        "--profile",
        default=os.getenv(
            "AWS_PROFILE",
            DEFAULT_PROFILE
        )
    )

    parser.add_argument(
        "--region",
        default=DEFAULT_REGION
    )

    parser.add_argument(
        "--days",
        type=int,
        default=DEFAULT_DAYS
    )

    parser.add_argument(
        "--output",
        default="AWS_Cost_IAM_Audit.xlsx"
    )

    args = parser.parse_args()

    print("\n======================================")
    print("       AWS COST + IAM AUDIT")
    print("======================================")

    # AWS SESSION
    try:

        session, identity = create_session(
            args.profile,
            args.region
        )

    except (ClientError, BotoCoreError) as exc:

        print("\nAWS authentication failed:")
        print(exc)

        return

    # DATE RANGE
    end_date = datetime.now(
        timezone.utc
    ).date()

    start_date = end_date - timedelta(
        days=args.days
    )

    start = start_date.isoformat()

    end = end_date.isoformat()

    print("\nAudit Period:")
    print("Start:", start)
    print("End:", end)
    print("Days:", args.days)

    # COST EXPLORER
    ce = get_cost_explorer(session)

    service_cost = get_service_cost(
        ce,
        start,
        end
    )

    daily_cost = get_daily_cost(
        ce,
        start,
        end
    )

    usage_cost = get_usage_breakdown(
        ce,
        start,
        end
    )

    operation_cost = get_operation_breakdown(
        ce,
        start,
        end
    )

    region_cost = get_region_cost(
        ce,
        start,
        end
    )

    forecast = get_cost_forecast(ce)

    explanations = explain_usage(
        usage_cost
    )

    # CURRENT RESOURCE INVENTORY
    # Currently scans the selected region.
    resources = get_current_resources(
        session,
        args.region
    )

    missing_tags = get_missing_owner_tags(
        resources
    )

    # KIRO DIAGNOSTIC
    kiro = kiro_diagnostic(
        session,
        identity,
        args.region
    )

    # CLOUDTRAIL IAM ATTRIBUTION
    cloudtrail_events = get_cloudtrail_activity(
        session,
        args.days
    )

    (
        iam_attribution,
        iam_timeline,
        unidentified_resources
    ) = attribute_resources(
        resources,
        cloudtrail_events
    )

    # GENERATE EXCEL
    generate_report(

        args.output,

        service_cost,

        daily_cost,

        usage_cost,

        operation_cost,

        region_cost,

        forecast,

        explanations,

        resources,

        missing_tags,

        kiro,

        iam_attribution,

        iam_timeline,

        unidentified_resources

    )

    print("\n======================================")
    print("          AUDIT COMPLETED")
    print("======================================")

    print("Service Cost Rows:", len(service_cost))

    print("Daily Cost Rows:", len(daily_cost))

    print("Usage Breakdown Rows:", len(usage_cost))

    print("Current Resources:", len(resources))

    print("CloudTrail Events:", len(cloudtrail_events))

    print("IAM Attribution Rows:", len(iam_attribution))

    print("Unidentified Resources:", len(unidentified_resources))

    print("\nExcel File:", args.output)


if __name__ == "__main__":

    main()


usage: ipykernel_launcher.py [-h] [--profile PROFILE] [--region REGION]
                             [--days DAYS] [--output OUTPUT]
ipykernel_launcher.py: error: unrecognized arguments: --f=/run/user/1000/jupyter/runtime/kernel-v30577c015bc63f7b8b1afd7ed1be1eb0aa88f9bcc.json


SystemExit: 2

/home/mukesh/Desktop/cost report/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
